In [1]:
# a) Fair value
import numpy as np
from scipy.stats import norm
from scipy.sparse import csc_matrix, eye, kron
from scipy.sparse.linalg import factorized

s0, r, sigma, T = 100.0, 0.05, 0.20, 1.0

def analytic_price(s1, s2, sigma, T):
    # max(S1, S2) = S2 + max(S1 - S2, 0)
    vol = np.sqrt(2) * sigma
    d1 = (np.log(s1 / s2) + 0.5 * vol**2 * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return s1 * norm.cdf(d1) + s2 * norm.cdf(-d2)

exact = analytic_price(s0, s0, sigma, T)
print(f"Analytic price: {exact:.6f}")

Analytic price: 111.246292


In [2]:
# b) PDE derivation
# Under the risk-neutral measure:
# dS1 = r*S1*dt + sigma*S1*dW1
# dS2 = r*S2*dt + sigma*S2*dW2
# Ito's formula and the discounted martingale condition give:
# V_t + r*S1*V_S1 + r*S2*V_S2
#     + (sigma**2/2)*(S1**2*V_S1S1 + S2**2*V_S2S2) - r*V = 0
# Terminal condition: V(T, S1, S2) = max(S1, S2)
# There is no mixed derivative because W1 and W2 are independent.

In [3]:
# c) Finite difference method (Crank-Nicolson)
# The following derivative matrices are taken from the lecture.
def get_d1(n):
    return 0.5 * (np.diag(np.ones(n - 1), 1) - np.diag(np.ones(n - 1), -1))

def get_d2(n):
    return np.diag(np.ones(n - 1), 1) - 2 * np.eye(n) + np.diag(np.ones(n - 1), -1)

def get_a(n, dx, r, sig):
    return (-r * np.eye(n)
            + (r - 0.5 * sig**2) / dx * get_d1(n)
            + 0.5 * sig**2 / dx**2 * get_d2(n))

def pde_price(s0, r, sigma, T, n=81, steps=240):
    x = np.linspace(-1.0, 1.0, n)
    h = x[1] - x[0]
    s = s0 * np.exp(x)
    m = n - 2

    L = csc_matrix(get_a(m, h, r, sigma) + r * np.eye(m))
    I = eye(m, format="csc")
    I2 = eye(m*m, format="csc")
    A = kron(L, I, format="csc") + kron(I, L, format="csc") - r * I2

    payoff = np.maximum(s[:, None], s[None, :])
    v = payoff[1:-1, 1:-1].ravel()

    a = sigma**2 / (2 * h**2)
    b = (r - sigma**2 / 2) / (2 * h)
    low, high = a - b, a + b
    edge = np.zeros((m, m))
    mid = s[1:-1]
    edge[0, :] += low * np.maximum(s[0], mid)
    edge[-1, :] += high * np.maximum(s[-1], mid)
    edge[:, 0] += low * np.maximum(mid, s[0])
    edge[:, -1] += high * np.maximum(mid, s[-1])

    dt = T / steps
    solve = factorized((I2 - dt/2 * A).tocsc())
    right = I2 + dt/2 * A
    for _ in range(steps):
        v = solve(right @ v + dt * edge.ravel())

    return float(v.reshape(m, m)[m//2, m//2])

for n, steps in [(41, 120), (81, 240)]:
    pde = pde_price(s0, r, sigma, T, n, steps)
    print(f"PDE ({n}x{n}): {pde:.6f}, error: {abs(pde - exact):.6f}")

PDE (41x41): 111.211571, error: 0.034720


PDE (81x81): 111.237656, error: 0.008636


In [4]:
# d) Monte Carlo comparison
rng = np.random.default_rng(2026)
z = rng.standard_normal((500_000, 2))
sT = s0 * np.exp((r - sigma**2 / 2) * T + sigma * np.sqrt(T) * z)
payoffs = np.exp(-r*T) * np.maximum(sT[:, 0], sT[:, 1])
mc = payoffs.mean()
se = payoffs.std(ddof=1) / np.sqrt(len(payoffs))

print(f"Analytic:    {exact:.6f}")
print(f"PDE:         {pde:.6f}")
print(f"Monte Carlo: {mc:.6f}")
print(f"MC 95% CI:  [{mc - 1.96*se:.6f}, {mc + 1.96*se:.6f}]")

Analytic:    111.246292
PDE:         111.237656
Monte Carlo: 111.261656
MC 95% CI:  [111.209715, 111.313597]
